# 🚦 Event-Driven Traffic Congestion Forecasting System
**Flipkart Grid Hackathon – Round 2**

This notebook is the **single entry-point** for the full pipeline.
Run cells top-to-bottom. No GPU required — CPU runtime is fine.

**Sections:**
1. Setup & Repo Clone
2. Data Upload & EDA
3. Preprocessing
4. Feature Engineering
5. Model Training & Evaluation
6. Recommendation Engine Demo
7. Simulated Live Feed
8. Streamlit Dashboard Launch (optional)
9. Push to GitHub (optional)

## 1. Setup — Install Dependencies & Clone Repo

In [ ]:
# Install all dependencies
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn streamlit pydeck shap joblib pyarrow pytest pyngrok -q
print('✅ Dependencies installed')

In [ ]:
import os, sys, importlib

REPO_URL = 'https://github.com/maidevalhoon/Flipkart_round2.git'
REPO_DIR = '/content/Flipkart_round2'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Always pull latest — avoids stale code from previous runs
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
os.environ['PROJECT_ROOT'] = REPO_DIR

# Ensure repo root is first on path
if REPO_DIR in sys.path:
    sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Clear Python's module cache so re-imports pick up the latest pulled code.
# Without this, Colab reuses old in-memory module objects even after git pull.
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]
importlib.invalidate_caches()

print(f'✅ Working directory: {os.getcwd()}')
print(f'✅ src/ modules cache cleared — will import fresh code')

## 2. Data Upload

In [ ]:
import os, shutil
from google.colab import files

RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
RAW_CSV = os.path.join(RAW_DIR, 'events.csv')

if not os.path.exists(RAW_CSV):
    print('Upload the CSV file: "Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"')
    uploaded = files.upload()
    # Rename whatever file was uploaded to events.csv
    for fname in uploaded:
        shutil.move(fname, RAW_CSV)
    print(f'✅ Saved to {RAW_CSV}')
else:
    print(f'✅ Raw CSV already present at {RAW_CSV}')

### 2b. EDA — Quick Dataset Summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
print(f'\nDate range: {raw["start_datetime"].min()} → {raw["start_datetime"].max()}')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

raw['event_cause'].value_counts().head(12).plot.barh(ax=axes[0,0], title='Event Causes')
raw['event_type'].value_counts().plot.bar(ax=axes[0,1], title='Event Type')
raw['priority'].value_counts().plot.bar(ax=axes[1,0], title='Priority (Target)', color=['tomato','steelblue'])
raw['corridor'].value_counts().head(10).plot.barh(ax=axes[1,1], title='Top Corridors')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nMissingness (top 15 columns):')
print((raw.isna().mean() * 100).sort_values(ascending=False).head(15).round(1).to_string())

## 3. Preprocessing

In [ ]:
from src.preprocessing import load_and_clean, save_clean
from src.config import CLEAN_PARQUET

clean_df = load_and_clean(RAW_CSV)
save_clean(clean_df, CLEAN_PARQUET)

print(f'\nClean shape: {clean_df.shape}')
print(f'Target balance:\n{clean_df["priority_high"].value_counts().rename({1:"High",0:"Low"})}')

In [ ]:
# Run preprocessing tests
!pytest tests/test_preprocessing.py -v --tb=short 2>&1 | tail -30

## 4. Feature Engineering & Time-Based Split

In [ ]:
from src.split import time_split
import pandas as pd

clean_df = pd.read_parquet(CLEAN_PARQUET)
train_df, val_df, test_df = time_split(clean_df)

In [ ]:
from src.features import build_features
import joblib, os
import numpy as np

# Build features for Model 1 (priority target)
builder, X_train, y_train, X_val, y_val, X_test, y_test = build_features(
    train_df, val_df=val_df, test_df=test_df
)

# Build y arrays for Model 2 (road closure target).
# Labels come directly from the dataframe column — no feature builder needed.
y_train_c = train_df["requires_road_closure_bool"].astype(int).values
y_val_c   = val_df["requires_road_closure_bool"].astype(int).values
y_test_c  = test_df["requires_road_closure_bool"].astype(int).values

# Save builder alongside model for dashboard use
os.makedirs('models', exist_ok=True)
joblib.dump(builder, 'models/severity_xgb_builder.pkl')

print(f'Train features : {X_train.shape}')
print(f'Val features   : {X_val.shape}')
print(f'Test features  : {X_test.shape}')
print(f'\nModel 1 (priority) — class balance:')
print(f'  Train: High={y_train.sum()} | Low={(y_train==0).sum()}')
print(f'\nModel 2 (closure) — class balance:')
print(f'  Train: TRUE={y_train_c.sum()} | FALSE={(y_train_c==0).sum()} ({y_train_c.mean()*100:.1f}% TRUE)')

In [ ]:
# Run feature engineering tests
!pytest tests/test_features.py -v --tb=short 2>&1 | tail -30

### ⚠️ Diagnostic: Why the previous model scored F1=1.0 (and why we fixed it)

`priority` in this dataset is assigned by police operators based almost entirely on corridor type:
- **Non-corridor** → 0% High (3122/3122 events)
- **Any named corridor** → 99–100% High (5029/5051 events)

A single `if corridor != "Non-corridor": predict High` achieves 99.9% accuracy — no ML needed.

**Fix applied:** Corridor target-encoding replaced with frequency encoding (log event count per corridor). This removes the trivial reconstruction of the label while keeping the spatial signal. The model now has to learn from event_cause + time + vehicle type + geo cluster, giving honest predictions (~0.75–0.88 F1).

We also add **Model 2 (road closure)** which is genuinely hard: 8.3% TRUE vs 91.7% FALSE, spread across both corridor types. This is more operationally useful ("does this event need a physical barricade?").

In [ ]:
# Diagnostic: corridor vs priority correlation (explains the F1=1.0 in previous run)
import pandas as pd

raw = pd.read_csv(RAW_CSV, low_memory=False)
corr_table = raw.groupby('corridor')['priority'].value_counts().unstack(fill_value=0)
corr_table['total'] = corr_table.sum(axis=1)
corr_table['pct_high'] = (corr_table.get('High', 0) / corr_table['total'] * 100).round(1)
print("Corridor → Priority breakdown:")
print(corr_table.sort_values('total', ascending=False).head(15).to_string())
print("\nConclusion: 'Non-corridor' = 0% High, named corridors = 99-100% High.")
print("Corridor alone predicts priority with 99.9% accuracy → target encoding was trivial.")
print("Fix: using frequency encoding (log event count) instead.")

## 5. Model Training & Evaluation

> ⚙️ **Compute:** CPU-only. Expected runtime < 60 seconds on Colab free tier.

In [ ]:
from src.model import train_and_evaluate

priority_model, p_threshold, closure_model, c_threshold, results = train_and_evaluate(
    X_train, y_train, X_val, y_val, X_test, y_test,
    X_train, y_train_c, X_val, y_val_c, X_test, y_test_c,
)

print(f'\n✅ Both models trained.')
print(f'\nModel 1 (Priority):')
print(f'  Threshold : {p_threshold:.3f}')
print(f'  Test F1   : {results["priority"]["test"]["f1_pos"]:.4f}')
print(f'  Test PR-AUC: {results["priority"]["test"]["prauc"]:.4f}')
if results.get("closure"):
    print(f'\nModel 2 (Road Closure):')
    print(f'  Threshold : {c_threshold:.3f}')
    print(f'  Test F1   : {results["closure"]["test"]["f1_pos"]:.4f}')
    print(f'  Test PR-AUC: {results["closure"]["test"]["prauc"]:.4f}')

In [ ]:
from IPython.display import Image, display
import os

print("=== Model 1: Priority Severity ===")
display(Image('models/priority_feature_importance.png'))
display(Image('models/priority_confusion_matrix.png'))

if os.path.exists('models/closure_feature_importance.png'):
    print("\n=== Model 2: Road Closure ===")
    display(Image('models/closure_feature_importance.png'))
    display(Image('models/closure_confusion_matrix.png'))

## 6. Recommendation Engine Demo

In [ ]:
!pytest tests/test_recommend.py -v --tb=short 2>&1 | tail -30

In [ ]:
from src.recommend import recommend, fallback_recommend

# Example 1: High-severity procession on Mysore Road during peak hour
r1 = recommend({
    'severity': 'High',
    'probability': 0.82,
    'event_cause': 'procession',
    'requires_road_closure': True,
    'is_corridor': 1,
    'event_type': 'planned',
    'hour': 20,
    'hour_bucket': 'evening',
    'dup_cluster_size': 5,
    'corridor_name': 'Mysore Road',
})
print('Example 1: Procession on Mysore Road')
print(f'  Manpower  : {r1["manpower_count"]}')
print(f'  Barricades: {r1["barricade_count"]} — {r1["barricade_placement"]}')
print(f'  Diversion : {r1["diversion_suggested"]} — {r1["diversion_note"]}')
print(f'  Rationale : {r1["rationale"]}')

print()

# Example 2: Low-severity breakdown, non-corridor
r2 = recommend({
    'severity': 'Low',
    'probability': 0.25,
    'event_cause': 'vehicle_breakdown',
    'requires_road_closure': False,
    'is_corridor': 0,
    'event_type': 'unplanned',
    'hour': 14,
    'hour_bucket': 'midday',
    'dup_cluster_size': 1,
    'corridor_name': 'Non-corridor',
})
print('Example 2: Vehicle breakdown, non-corridor')
print(f'  Manpower  : {r2["manpower_count"]}')
print(f'  Barricades: {r2["barricade_count"]}')
print(f'  Diversion : {r2["diversion_suggested"]}')

# Fallback demo
print('\nFallback (model unavailable):')
rf = fallback_recommend('congestion', requires_road_closure=True)
print(f'  Severity  : {rf["severity"]}')
print(f'  Manpower  : {rf["manpower_count"]}')
print(f'  Rationale : {rf["rationale"]}')

## 7. Simulated Live Feed (Test Window Replay)

In [ ]:
import pandas as pd
from src.realtime_sim import simulate_stream
from src.split import time_split

# Re-load test window
clean_df = pd.read_parquet(CLEAN_PARQUET)
_, _, test_df = time_split(clean_df)

print('⚠️  SIMULATED LIVE FEED — historical test-window events replayed')
print('='*60)

sim_results = []
for ev in simulate_stream(test_df, builder, priority_model, p_threshold, delay=0, n=25):
    sim_results.append(ev)

# Summary table
import pandas as pd
summary = pd.DataFrame([{
    'Time': e['timestamp'][:19],
    'Cause': e['event_cause'],
    'Predicted': e['predicted_severity'],
    'Actual': e['actual_priority'],
    'Prob': f"{e['probability']:.3f}",
    'Manpower': e['recommendation']['manpower_count'],
    'Barricades': e['recommendation']['barricade_count'],
    'Diversion': 'Yes' if e['recommendation']['diversion_suggested'] else 'No',
} for e in sim_results])

# Accuracy on these 25
correct = (summary['Predicted'] == summary['Actual']).sum()
print(f'\nAccuracy on replay: {correct}/{len(summary)} = {correct/len(summary)*100:.0f}%')
display(summary)

### 7b. Write feedback_log.csv from simulation results

`feedback_log.csv` is the post-event learning schema. This cell creates it directly from the simulated predictions above — so it exists whether or not the Streamlit dashboard is running.

In [ ]:
import pandas as pd
import datetime
import os

FEEDBACK_LOG = os.path.join(REPO_DIR, 'feedback_log.csv')

# Build feedback rows from sim_results (predicted cols filled; actual cols left blank for operator)
feedback_rows = []
for ev in sim_results:
    rec = ev['recommendation']
    feedback_rows.append({
        'event_id':                ev['row_idx'],
        'timestamp':               ev['timestamp'],
        'event_cause':             ev['event_cause'],
        'predicted_severity':      ev['predicted_severity'],
        'predicted_prob':          round(ev['probability'], 4),
        'recommended_manpower':    rec['manpower_count'],
        'recommended_barricades':  rec['barricade_count'],
        'diversion_suggested':     rec['diversion_suggested'],
        # ── operator fills these after the event ──────────────────────────────
        'actual_severity':         '',   # fill: High / Low
        'actual_resolution_mins':  '',   # fill: integer minutes
        'actual_manpower_used':    '',   # fill: integer
        'operator_override':       '',   # fill: yes / no
        'notes':                   '',
        'logged_at':               datetime.datetime.now().isoformat(),
    })

feedback_df = pd.DataFrame(feedback_rows)
feedback_df.to_csv(FEEDBACK_LOG, index=False)

print(f'✅ feedback_log.csv written → {FEEDBACK_LOG}')
print(f'   {len(feedback_df)} rows  |  {len(feedback_df.columns)} columns')
print(f'\nSchema:')
print(feedback_df.dtypes.to_string())
print(f'\nFirst 3 rows (actual_* columns left blank for operator):')
display(feedback_df.head(3))

## 8. Streamlit Dashboard (Optional — Colab with ngrok)

Requires a free ngrok account: https://ngrok.com/
Set your token below, then run the cell.

In [ ]:
NGROK_TOKEN = ''  # ← paste your ngrok authtoken here

if NGROK_TOKEN:
    !pip install pyngrok -q
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    !streamlit run app/dashboard.py &
    import time; time.sleep(3)
    tunnel = ngrok.connect(8501)
    print(f'\n🌐 Dashboard URL: {tunnel.public_url}')
else:
    print('Set NGROK_TOKEN above to launch the Streamlit dashboard.')
    print('Skipping dashboard launch.')

## 9. Push Results to GitHub (Optional)

In [ ]:
# Configure git identity (required for first commit)
!git config user.email 'you@example.com'
!git config user.name 'Your Name'

# Stage generated artifacts
!git add models/metrics.json models/feature_importance.png models/confusion_matrix.png \
         eda_overview.png feedback_log.csv 2>/dev/null; echo done

# Commit
!git commit -m 'Add trained model artifacts and evaluation plots' || echo 'Nothing to commit'

# Push (will prompt for credentials if HTTPS — use a PAT token)
!git push origin main || git push origin master || echo 'Push failed — check credentials'